In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Data Engineering\Projects\fleetpulse_project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [17]:
from scripts.utils.spark_session import create_spark_session
from scripts.utils.config import (
    BRONZE_BASE_PATH,
    SILVER_BASE_PATH
)
import pyspark.sql.functions as F 
from pyspark.sql.window import Window

In [18]:
table = "equipment_categories"
spark = create_spark_session("DataFrameAPIPractise")
bronze_path =  f"{BRONZE_BASE_PATH}/{table}"

In [19]:
df = spark.read.parquet(bronze_path)
df.printSchema()

root
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- parent_category_id: long (nullable = true)
 |-- equipment_class: string (nullable = true)
 |-- equipment_type: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- source_table: string (nullable = true)
 |-- bronze_load_type: string (nullable = true)



In [20]:
df.show(truncate=False)

+-----------+-----------------+----------------------+------------------+---------------+--------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+--------------------+----------------+
|category_id|category_code    |category_name         |parent_category_id|equipment_class|equipment_type|source_system|created_at               |updated_at               |is_deleted|ingestion_timestamp       |ingestion_date|source_table        |bronze_load_type|
+-----------+-----------------+----------------------+------------------+---------------+--------------+-------------+-------------------------+-------------------------+----------+--------------------------+--------------+--------------------+----------------+
|1          |CAT-EARTH        |Earthmoving           |NULL              |Heavy Equipment|Category      |RENTAL_OLTP  |2026-07-21 21:48:05.66039|2026-07-21 21:48:05.66039|false     |2026-08-04 17:47:06.740783|2026-0

In [21]:
df_clean = (df
            .withColumn("category_code", F.upper(F.col("category_code")))
            .withColumn("source_system", F.upper(F.col("source_system")))
            .withColumn("source_system", F.upper(F.col("bronze_load_type")))
            .withColumn("category_name", F.trim(F.col("category_name")))
            .withColumn("equipment_class", F.trim(F.col("equipment_class")))
            .withColumn("equipment_type", F.trim(F.col("equipment_type")))
            .withColumn("source_table", F.trim(F.col("source_table")))
            .withColumn("is_deleted", F.coalesce(F.col("source_table"),F.lit(False)))
        )

In [22]:
window_spec = (
                Window
                .partitionBy("category_id")
                .orderBy(
                    F.col("updated_at").desc(),
                    F.col("ingestion_timestamp").desc())
            )

df_dedup = (
            df_clean
            .withColumn("row_num", F.row_number().over(window_spec))
            .filter(F.col("row_num") == 1)
            .drop(F.col("row_num"))
            )

In [23]:
df_hash = (
        df_dedup
        .withColumn("row_hash", 
                    F.sha2(
                        F.concat_ws(
                                    "|",
                                    F.col("category_code"),
                                    F.col("category_name"),
                                    F.col("parent_category_id"),
                                    F.col("equipment_class"),
                                    F.col("equipment_type"),
                                    F.col("is_deleted")
                        ),256
                    ))    
)

In [24]:
silver_df = (
                df_hash
                .withColumn("silver_created_timestamp", F.current_timestamp())
                .withColumn("silver_updated_timestamp", F.current_timestamp())
                .withColumn("silver_quality_status", F.lit("VALID"))
            )

In [27]:
print(f"Bronze path:{bronze_path}")
#print(f"Silver Path: {silver_path}")
print("Bronze count:", df.count())
print("Silver Count", silver_df.count())
distint_customer_count = silver_df.select("category_id").distinct().count()
print(f"Distint customer count : {distint_customer_count}")

Bronze path:s3a://fleetpulse-lake/bronze/rental_oltp/equipment_categories
Bronze count: 8
Silver Count 8
Distint customer count : 8
